# Bootstrap & Preflight — 07-CNN.ipynb

**What this does**
- Verifies expected upstream notebooks have been executed and essential artefact folders exist.
- Prints clear guidance to run prerequisites if required files/folders are missing.

**Upstream prerequisites (recommended order)**
- `01-Setup_Preflight.ipynb`
- `02-Feature_Engineering.ipynb`

**Checks performed**
- Confirms `DATA_PATH` exists (from Section 0.1).
- Ensures `staging/` and `out/` directories exist when required downstream.
- Provides actionable instructions when a check fails.


In [ ]:
# =====================================================
# 7.0 — Load raw payloads and prepare byte-token sequences
#   Uses persisted splits from 01–02, reads 'payload' column.
#   Tokenization: map each byte [0..255] to an integer id (0..255), with 256 = PAD.
# =====================================================
from pathlib import Path
import json, numpy as np, pandas as pd

OUT_ROOT   = Path(globals().get("OUT_ROOT","out"))
SPLITS_DIR = Path(globals().get("SPLITS_DIR", OUT_ROOT/"splits"))
TARGET_COL = globals().get("TARGET_COL", "label")

# --- Load splits ---
def _load_y(p, col="label"):
    df = pd.read_parquet(p)
    c = col if col in df.columns else df.columns[0]
    return np.asarray(df[c]).ravel().astype(int)

Xtr_df = pd.read_parquet(SPLITS_DIR/"X_train.parquet")
Xva_df = pd.read_parquet(SPLITS_DIR/"X_val.parquet")
y_train = _load_y(SPLITS_DIR/"y_train.parquet", TARGET_COL)
y_val   = _load_y(SPLITS_DIR/"y_val.parquet", TARGET_COL)

# --- Guards ---
assert "payload" in Xtr_df.columns and "payload" in Xva_df.columns, \
    "[7.0] 'payload' column is required in X_train.parquet / X_val.parquet (produced in 01–02)."
assert Xtr_df.shape[0] == len(y_train) and Xva_df.shape[0] == len(y_val), \
    "[7.0] X/y length mismatch."

# --- Byte tokenizer (simple & deterministic) ---
# Map each string payload to np.uint16 array of byte IDs: 0..255; PAD=256
PAD_ID     = 256
VOCAB_SIZE = 257   # 256 byte values + PAD
MAX_LEN    = int(globals().get("CNN_MAX_LEN", 512))  # truncate/clip length

def payload_to_ids(s: str, max_len: int=MAX_LEN) -> np.ndarray:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    b = s.encode("utf-8", errors="ignore")[:max_len]  # bytes
    out = np.full((max_len,), PAD_ID, dtype=np.uint16)
    if len(b):
        out[:len(b)] = np.frombuffer(b, dtype=np.uint8)
    return out

def batch_encode(series: pd.Series, max_len: int=MAX_LEN) -> np.ndarray:
    arr = np.stack([payload_to_ids(s, max_len) for s in series.astype(str)], axis=0)
    return arr

X_train_ids = batch_encode(Xtr_df["payload"], MAX_LEN)
X_val_ids   = batch_encode(Xva_df["payload"], MAX_LEN)

print(f"[7.0] Sequences ready: train {X_train_ids.shape}, val {X_val_ids.shape}, vocab={VOCAB_SIZE}, pad_id={PAD_ID}")
print(f"[7.0] Label counts — train:{dict(zip(*np.unique(y_train, return_counts=True)))}  val:{dict(zip(*np.unique(y_val, return_counts=True)))}")


>>> Section 7: start
[ok] Loaded/constructed payload sequence features (sparse)
[ready] CNN inputs available


## Section 7.1 — Section

In [ ]:
# =====================================================
# 7.1 — Build & train a 1D CNN over byte-token sequences
#   Architecture: Embedding → Conv1D blocks → GlobalMaxPool → Dense
#   Loss: Binary cross-entropy; Metric: AUPRC (Average Precision)
# =====================================================
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import average_precision_score
import numpy as np
from pathlib import Path

RANDOM_STATE = int(globals().get("RANDOM_STATE", 42))
tf.keras.utils.set_random_seed(RANDOM_STATE)

EMB_DIM     = int(globals().get("CNN_EMB_DIM", 64))
NUM_FILTERS = int(globals().get("CNN_NUM_FILTERS", 128))
KERNEL_S    = int(globals().get("CNN_KERNEL_SIZE", 5))
DROPOUT     = float(globals().get("CNN_DROPOUT", 0.2))
LR          = float(globals().get("CNN_LR", 1e-3))
BATCH_SIZE  = int(globals().get("CNN_BATCH_SIZE", 256))
EPOCHS      = int(globals().get("CNN_EPOCHS", 8))  # tune as needed
PATIENCE    = int(globals().get("CNN_EARLY_STOP", 2))

def make_model(vocab_size=VOCAB_SIZE, pad_id=PAD_ID, max_len=MAX_LEN):
    inp = layers.Input(shape=(max_len,), dtype="int32")
    x = layers.Embedding(input_dim=vocab_size, output_dim=EMB_DIM, mask_zero=False,
                         input_length=max_len, name="byte_embedding")(inp)
    # Conv block 1
    x = layers.Conv1D(NUM_FILTERS, KERNEL_S, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(pool_size=2)(x)
    x = layers.Dropout(DROPOUT)(x)
    # Conv block 2
    x = layers.Conv1D(NUM_FILTERS, KERNEL_S, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalMaxPool1D()(x)
    x = layers.Dropout(DROPOUT)(x)
    # Dense head
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out, name="cnn_byte_ids")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="AUPRC"), keras.metrics.AUC(name="AUROC")]
    )
    return model

cnn = make_model()

# Training setup
OUT_MODELS = Path(globals().get("OUT_ROOT","out")) / "models"
OUT_MODELS.mkdir(parents=True, exist_ok=True)
CKPT = OUT_MODELS / "cnn_payload.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=str(CKPT), monitor="val_AUPRC",
                                    save_best_only=True, mode="max", verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_AUPRC", patience=PATIENCE,
                                  mode="max", restore_best_weights=True, verbose=1)
]

history = cnn.fit(
    X_train_ids, y_train,
    validation_data=(X_val_ids, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

# Evaluate on validation split with Average Precision (sklearn)
y_val_pred = cnn.predict(X_val_ids, batch_size=BATCH_SIZE).ravel()
cnn_val_ap = float(average_precision_score(y_val, y_val_pred))
print(f"[7.1] CNN Validation AP: {cnn_val_ap:.4f}")

# Save final model (best checkpoint is already saved by ModelCheckpoint)
cnn.save(CKPT, overwrite=True)
print(f"[7.1] Saved CNN model → {CKPT}")


IndentationError: unexpected indent (4035798116.py, line 5)

## Section 7.2 — Section

In [ ]:
# =====================================================
# 7.2 — Persist cnn.json so 09 can pick up the benchmark
# =====================================================
import time, json
from pathlib import Path

OUT_ROOT   = Path(globals().get("OUT_ROOT","out"))
OUT_REPORT = OUT_ROOT / "reports"
OUT_REPORT.mkdir(parents=True, exist_ok=True)

cnn_report = {
    "model": "cnn_payload",
    "status": "ok",
    "reason": "",
    "metrics": {"val_average_precision": cnn_val_ap},
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "artifact": "models/cnn_payload.keras",
    "data": {
        "max_len": int(globals().get("CNN_MAX_LEN", 512)),
        "vocab_size": 257,
        "pad_id": 256,
    },
}
(OUT_REPORT / "cnn.json").write_text(json.dumps(cnn_report, indent=2))
print(f"[7.2] wrote {OUT_REPORT/'cnn.json'}")


>>> Section 7.2: CNN Performance Analysis
[warn] CNN artefact not found; skipping CNN scoring. Train in Section 7 to enable a direct comparison.


#### Available comparison (from `compare_summary.csv`)

,family,variant,baseline_AP,tuned_AP,delta_AP,n_val
0,LGBM,LGBM Grid,0.744378,0.749121,0.004743,15977
1,LGBM,LGBM Random,0.744378,0.748766,0.004388,15977
2,LGBM,LGBM Bayes,0.744378,0.741136,-0.003242,15977
3,LGBM,LGBM Warm,0.744378,0.741136,-0.003242,15977
4,XGB vs LGBM,XGB Baseline vs Lgbm Grid,0.741678,0.749121,0.007443,15977


### 7.2b — Persist CNN metric for downstream notebooks

Persists validation accuracy/AP from 7.2 into `out/metrics/cnn.json` so 08/09 can autowire comparisons.

In [ ]:
# --- 7.2b: persist CNN metric ---
from pathlib import Path
import json

if "CNN_VAL_ACC" in globals():
    out_dir = Path(globals().get("OUT_ROOT", "out"))
    metrics_dir = out_dir / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)
    path = metrics_dir / "cnn.json"
    path.write_text(json.dumps({"val_acc": float(CNN_VAL_ACC)}, indent=2))
    print(f"[persist] wrote CNN metric → {path}")
else:
    print("[persist] CNN_VAL_ACC not in globals — skipping write.")

## Section 7.3 — Section

In [ ]:
# =====================================================
# Section 7.3 — Section
# =====================================================
print(">>> Section 7.3: start")
except Exception:
    pd = None
try:
except Exception:
    joblib = None
try:
except Exception:
    sp = None
# [removed duplicate canonical assignment: OUT_ROOT]
# [removed duplicate canonical assignment: STAGE_ROOT]
reports_dir = OUT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir = OUT_ROOT / 'models'
lgbm_candidates = [models_dir / 'lgbm_best.joblib', models_dir / 'lgbm_champion.joblib', STAGE_ROOT / 'models' / 'lgbm_best.joblib']
cnn_candidates = [models_dir / 'cnn_benchmark.keras', models_dir / 'cnn_baseline.keras', STAGE_ROOT / 'models' / 'cnn_best.h5']
lgbm_path = next((p for p in lgbm_candidates if p.exists()), None)
cnn_path = next((p for p in cnn_candidates if p.exists()), None)
split_dir = STAGE_ROOT / 'split_preproc'
Xtr_npz = split_dir / 'X_train_t.npz'
Xva_npz = split_dir / 'X_val_t.npz'
ytr_npy = split_dir / 'y_train.npy'
yva_npy = split_dir / 'y_val.npy'
seq_dir = STAGE_ROOT / 'payload_seq_preproc'
seq_tr = seq_dir / 'payload_seq_train.npz'
seq_va = seq_dir / 'payload_seq_val.npz'

def _load_npz_dense_or_sparse(p: Path):
    if p.suffix == '.npz':
        if sp is not None:
            try:
                return sp.load_npz(p)
            except Exception:
                pass
        return np.load(p)['arr']
    raise ValueError(f'Unsupported format: {p}')

def _ensure_features():
    """Return (Xtr_tree, Xva_tree, y_train, y_val, Xtr_seq, Xva_seq) where _seq may be None."""
    if not all((p.exists() for p in [Xtr_npz, Xva_npz, ytr_npy, yva_npy])):
        return None
    Xtr_tree = _load_npz_dense_or_sparse(Xtr_npz)
    Xva_tree = _load_npz_dense_or_sparse(Xva_npz)
    y_train = np.load(ytr_npy)
    y_val = np.load(yva_npy)
    Xtr_seq = Xva_seq = None
    if seq_tr.exists() and seq_va.exists() and (sp is not None):
        Xtr_seq = sp.load_npz(seq_tr)
        Xva_seq = sp.load_npz(seq_va)
    return (Xtr_tree, Xva_tree, y_train, y_val, Xtr_seq, Xva_seq)

def _bin_metrics(y_true, y_score, tau=None):
    ap = float(average_precision_score(y_true, y_score))
    if tau is None:
        p, r, thr = precision_recall_curve(y_true, y_score)
        best_f1, best_tau = (0.0, 0.5)
        for i, t in enumerate(thr):
            denom = p[i + 1] + r[i + 1]
            f = 0.0 if denom == 0 else 2 * p[i + 1] * r[i + 1] / denom
            if f > best_f1:
                best_f1, best_tau = (f, float(t))
        tau = best_tau
    y_hat = (y_score >= tau).astype(int)
    cm = confusion_matrix(y_true, y_hat).ravel().tolist()
    return {'ap': ap, 'tau': float(tau), 'f1': float(f1_score(y_true, y_hat)), 'precision': float(precision_score(y_true, y_hat, zero_division=0)), 'recall': float(recall_score(y_true, y_hat, zero_division=0)), 'cm': cm}

def _metrics_from_reports():
    """Recover LightGBM-ish metrics from existing reports, best-effort."""
    champ_p = reports_dir / 'champion.json'
    if champ_p.exists():
        try:
            ch = json.loads(champ_p.read_text())
            ap = ch.get('ap') or ch.get('pr_auc') or ch.get('average_precision')
            f1 = ch.get('f1')
            tau = ch.get('tau') or ch.get('threshold') or ch.get('threshold_tau')
            precision = ch.get('precision')
            recall = ch.get('recall')
            cm = ch.get('cm') or ch.get('confusion_matrix')
            if ap is not None and f1 is not None:
                return {'ap': float(ap), 'f1': float(f1), 'tau': float(tau) if tau is not None else 'N/A', 'precision': float(precision) if precision is not None else np.nan, 'recall': float(recall) if recall is not None else np.nan, 'cm': cm if cm is not None else 'N/A'}
        except Exception:
            pass
    comp_p = reports_dir / 'compare_summary.csv'
    if pd is not None and comp_p.exists():
        try:
            df = pd.read_csv(comp_p)
            model_col = next((c for c in df.columns if 'variant' in c.lower() or 'model' in c.lower()), None)
            ap_col = next((c for c in df.columns if c.lower() in ('tuned_ap', 'ap', 'pr_auc', 'average_precision')), None)
            if model_col and ap_col:
                row = df[df[model_col].astype(str).str.contains('LGBM', case=False, na=False)].head(1)
                if not row.empty:
                    return {'ap': float(row.iloc[0][ap_col]), 'f1': np.nan, 'tau': 'N/A', 'precision': np.nan, 'recall': np.nan, 'cm': 'N/A'}
        except Exception:
            pass
    km_p = reports_dir / 'key_metrics.csv'
    if pd is not None and km_p.exists():
        try:
            df = pd.read_csv(km_p)
            ap_col = next((c for c in df.columns if c.lower() in ('ap', 'pr_auc', 'average_precision')), None)
            f1_col = next((c for c in df.columns if c.lower() in ('f1', 'f1_score')), None)
            tau_col = next((c for c in df.columns if 'tau' in c.lower() or 'threshold' in c.lower()), None)
            if ap_col:
                return {'ap': float(df.iloc[0][ap_col]), 'f1': float(df.iloc[0][f1_col]) if f1_col else np.nan, 'tau': float(df.iloc[0][tau_col]) if tau_col and (not pd.isna(df.iloc[0][tau_col])) else 'N/A', 'precision': np.nan, 'recall': np.nan, 'cm': 'N/A'}
        except Exception:
            pass
    return None
rows = []
features = _ensure_features()
tree_row = None
if lgbm_path is not None and joblib is not None and (features is not None):
    Xtr_tree, Xva_tree, y_train, y_val, Xtr_seq, Xva_seq = features
    try:
        lgbm = joblib.load(lgbm_path)
        p_tree = lgbm.predict_proba(Xva_tree)[:, 1]
        m = _bin_metrics(y_val, p_tree, tau=None)
        tree_row = {'model': 'LightGBM (tree)', 'ap': m['ap'], 'f1_at_tau': m['f1'], 'tau_star': m['tau'], 'precision': m['precision'], 'recall': m['recall'], 'cm_tn_fp_fn_tp': m['cm']}
        print(f'[ok] Scored LightGBM from {lgbm_path.name}')
    except Exception as e:
        print(f'[warn] Live LightGBM scoring failed: {e}')
if tree_row is None:
    m = _metrics_from_reports()
    if m is not None:
        tree_row = {'model': 'LightGBM (report)', 'ap': m['ap'], 'f1_at_tau': m['f1'], 'tau_star': m['tau'], 'precision': m['precision'], 'recall': m['recall'], 'cm_tn_fp_fn_tp': m['cm']}
        print('[ok] Recovered LightGBM metrics from reports.')
    else:
        tree_row = {'model': 'LightGBM (missing)', 'ap': np.nan, 'f1_at_tau': np.nan, 'tau_star': 'N/A', 'precision': np.nan, 'recall': np.nan, 'cm_tn_fp_fn_tp': 'N/A'}
        print('[warn] Could not locate LightGBM artefact or metrics; emitting N/A row.')
rows.append(tree_row)
if cnn_path is None:
    rows.append({'model': 'CNN (missing)', 'ap': np.nan, 'f1_at_tau': np.nan, 'tau_star': 'N/A', 'precision': np.nan, 'recall': np.nan, 'cm_tn_fp_fn_tp': 'N/A'})
    print('[warn] CNN artefact not found; emitting N/A row.')
else:
    try:
        cnn = keras.models.load_model(cnn_path)
        X_seq = None
        if features is not None:
            _, Xva_tree, _, y_val, _, Xva_seq = features
            X_seq = Xva_seq if Xva_seq is not None else Xva_tree
        elif seq_va.exists() and sp is not None:
            X_seq = sp.load_npz(seq_va)
            y_val = np.load(yva_npy) if yva_npy.exists() else None
        elif Xva_npz.exists():
            X_seq = _load_npz_dense_or_sparse(Xva_npz)
            y_val = np.load(yva_npy) if yva_npy.exists() else None
        if X_seq is None or y_val is None:
            raise FileNotFoundError('Validation features/labels unavailable for CNN scoring.')
        p_cnn = cnn.predict(X_seq, verbose=0).reshape(-1)
        m = _bin_metrics(y_val, p_cnn, tau=None)
        rows.append({'model': f'CNN ({cnn_path.name})', 'ap': m['ap'], 'f1_at_tau': m['f1'], 'tau_star': m['tau'], 'precision': m['precision'], 'recall': m['recall'], 'cm_tn_fp_fn_tp': m['cm']})
        print(f'[ok] Scored CNN from {cnn_path.name}')
    except Exception as e:
        print(f'[warn] CNN scoring failed: {e}')
        rows.append({'model': f'CNN ({cnn_path.name})', 'ap': np.nan, 'f1_at_tau': np.nan, 'tau_star': 'N/A', 'precision': np.nan, 'recall': np.nan, 'cm_tn_fp_fn_tp': 'N/A'})
if pd is not None:
    df = pd.DataFrame(rows)
    df.to_csv(reports_dir / 'compare_tree_vs_cnn.csv', index=False)
    display(Markdown('#### Tree vs CNN — comparison (no-artefact safe fallback)'))
    display(df)
else:
    (reports_dir / 'compare_tree_vs_cnn.json').write_text('\n'.join((json.dumps(r) for r in rows)), encoding='utf-8')
print(f"[ok] Comparison persisted → {reports_dir / 'compare_tree_vs_cnn.csv'}")

>>> Section 7.3: Comparison of Optimised LightGBM vs CNN
[ok] Recovered LightGBM metrics from reports.
[warn] CNN artefact not found; emitting N/A row.


#### Tree vs CNN — comparison (no-artefact safe fallback)

,model,ap,f1_at_tau,tau_star,precision,recall,cm_tn_fp_fn_tp
0,LightGBM (report),0.749121,NaN,N/A,NaN,NaN,N/A
1,CNN (missing),NaN,NaN,N/A,NaN,NaN,N/A


[ok] Comparison persisted → out/reports/compare_tree_vs_cnn.csv


# Section 7.1 — Section

In [ ]:
# =====================================================
# Section 7.1 — Section
# =====================================================
print(">>> Section 7.1: start")
try:
    _res_var = res if 'res' in locals() else results if 'results' in locals() else []
    # Prefer existing proba/pred; else derive via estimator if available
    if 'proba' in locals():
        _scores = proba
        _pred = (proba >= 0.5).astype(int)
    elif 'clf' in locals():
        _Xval = X_va_t if 'X_va_t' in locals() else X_val if 'X_val' in locals() else X_va if 'X_va' in locals() else None
        _scores, _pred = get_scores_and_pred(clf, _Xval)
    elif 'model' in locals():
        _Xval = X_va_t if 'X_va_t' in locals() else X_val if 'X_val' in locals() else X_va if 'X_va' in locals() else None
        _scores, _pred = get_scores_and_pred(model, _Xval)
    else:
        raise RuntimeError('no proba/estimator available for metrics hook')
    _fold = fold if 'fold' in locals() else 1
    _res_rec = fold_report("6.3", _fold, y_va, _scores, _pred)
    # Keep results list in 'res'
    res = res if 'res' in locals() else []
    res.append(_res_rec)
except Exception as _e:
    print(f"[warn] metrics hook skipped: {_e}")

# --- metrics summary (standard) ---
try:
    summary_report("6.3", res if 'res' in locals() else results if 'results' in locals() else [])
except Exception as _e:
    print(f"[warn] summary skipped: {_e}")